# 02 - Liquidity Diagnostics
Compute spread, depth, quote presence, resilience and tracking-error metrics.

In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent dir to sys.path to import framework
sys.path.append(os.path.abspath('..'))

from framework.metrics import (
    compute_depth_profile, 
    detect_volatility_spikes, 
    spread_resilience_minutes, 
    price_tracking_error_pct, 
    summarize_metrics,
    compute_spread_metrics
)

sns.set_theme(style="whitegrid")

In [ ]:
try:
    # Using Underscored names for Tokocrypto as fixed in collection script
    tokobtc = pd.read_csv('../data/raw/orderbook_tokocrypto_BTC_USDT.csv')
    binbtc = pd.read_csv('../data/raw/orderbook_binance_BTCUSDT.csv')
    kline = pd.read_csv('../data/raw/klines_tokocrypto_BTC_USDT_5m.csv')
    
    print(f"Loaded {len(tokobtc)} Tokocrypto snapshots, {len(binbtc)} Binance snapshots")
    
    # Basic Liquidity Diagnostics
    depth = compute_depth_profile(tokobtc)
    spikes = detect_volatility_spikes(kline)
    spike_times = spikes.loc[spikes['is_spike'], 'open_time_utc']
    
    res = spread_resilience_minutes(tokobtc, spike_times)
    tracking = price_tracking_error_pct(tokobtc, binbtc)
    
    summary = summarize_metrics(tokobtc, depth_df=depth, resilience_minutes=res, tracking_df=tracking)
    
    # Table Display
    summary_df = pd.Series(summary).to_frame(name='Metric Value')
    display(summary_df)
except Exception as e:
    print(f"Warning: Could not process all metrics. Data may be insufficient. Error: {e}")

In [ ]:
if 'tokobtc' in locals():
    # Spread Time-Series
    plot_data = compute_spread_metrics(tokobtc)
    plt.figure(figsize=(12, 5))
    plt.plot(pd.to_datetime(plot_data['timestamp_utc']), plot_data['spread_pct'], marker='o', linestyle='-', color='royalblue')
    plt.title('Tokocrypto BTC/USDT Bid-Ask Spread (%)')
    plt.ylabel('Spread %')
    plt.grid(True, alpha=0.3)
    plt.show()